In [2]:
import matplotlib.pyplot as plt
from nilearn.connectome import ConnectivityMeasure
from brainspace.gradient import GradientMaps
from brainspace.utils.parcellation import map_to_labels
import numpy as np
import nibabel as nib
from nilearn import datasets
import os.path as op
import os
from nilearn import signal
import pandas as pd

bids_folder = '/Users/mrenke/data/ds-stressrisk'
bids_folder_ = '/Volumes/mrenkeED/data/ds-stressrisk'
target_folder = op.join(bids_folder_,'derivatives','correlation_matrices')

subList = ['01','02','03','04','05','08','09','10','11','12','13','14'] #
ses = 1
specification=''

In [25]:
from brainspace.datasets.base import load_parcellation

atlas = datasets.fetch_atlas_surf_destrieux() # len(atlas.map_left) = 10242 , as fsaverage5
atlas_s = load_parcellation('schaefer',scale=100) # len(atlas_s[0]) = 32492, as conte69


In [3]:
#gm = np.load('/Volumes/mrenkeED/data/ds-stressrisk/derivatives/correlation_matrices/cm_av1-4_unfiltered.npy')
gm = np.load('/Volumes/mrenkeED/data/ds-stressrisk/derivatives/correlation_matrices/cm_av50_unfiltered.npy')


In [11]:
np.shape(gm)[1]

3

In [12]:
atlas = datasets.fetch_atlas_surf_destrieux() # len(atlas.map_left) = 10242 , as fsaverage5
regions = atlas['labels'].copy()
masked_regions = [b'Medial_wall', b'Unknown']
masked_labels = [regions.index(r) for r in masked_regions] # [42, 0]

labeling = np.concatenate([atlas['map_left'], atlas['map_right']]) # atlas['map_left'] == atlas.map_left -> array, each vertex has a label assignment (a number from 0-51)
mask = ~np.isin(labeling, masked_labels)

# Map gradients to original parcels
grad = [None] * np.shape(gm)[1]
for i, g in enumerate(gm.T):
    grad[i] = map_to_labels(g, labeling, mask=mask, fill=np.nan)


In [14]:
from brainspace.datasets import load_fsa5
from brainspace.plotting import plot_hemispheres

surf_lh, surf_rh = load_fsa5()

# sphinx_gallery_thumbnail_number = 2
plot_hemispheres(surf_lh, surf_rh, array_name=grad, size=(1200, 400), cmap='viridis_r',
                 color_bar=True, label_text=['Grad1', 'Grad2', 'Grad3'], zoom=1.5)



: 

: 

In [3]:
target_folder = op.join(bids_folder_,'derivatives','correlation_matrices')
sub='01'
filtered = 'unfiltered'

file_name = op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_{filtered}.csv')
cm = pd.read_csv(file_name, index_col=0)
cm.shape

(18715, 18715)

In [8]:
from utils import cleanTS
from scipy.sparse.csgraph import connected_components

target_folder = op.join(bids_folder_,'derivatives','correlation_matrices')

for sub in subList:
    mask = ~np.isin(labeling, masked_labels)
    clean_ts = cleanTS(sub, ses,bids_folder=bids_folder)
    seed_ts = clean_ts[mask]

    # filter out nodes that are not connected to the rest
    correlation_measure = ConnectivityMeasure(kind='correlation')
    graph = correlation_measure.fit_transform([seed_ts.T])[0] #correlation_matrix_noParcel
    pd.DataFrame(graph).to_csv(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_unfiltered.csv')) # .shape = (18715, 18715)
    corr_matrices_unfiltered.append(pd.DataFrame(graph))

    cc = connected_components(graph)
    mask_cc = cc[1] == 0 # all nodes in 0 belong to the largest connected component, check #-components in cc[0]
    mask[mask == True] = mask_cc
    seed_ts = clean_ts[mask]   

    correlation_measure = ConnectivityMeasure(kind='correlation')
    correlation_matrix = correlation_measure.fit_transform([seed_ts.T])[0]
    pd.DataFrame(correlation_matrix).to_csv(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_onlyconncomp.csv')) # .shape = (18709, 18709)
    
    print(sub, 'done')
    

    
   

10 done


In [4]:
# one single subjects
from utils import cleanTS
from scipy.sparse.csgraph import connected_components

sub = '09'

target_folder = op.join(bids_folder_,'derivatives','correlation_matrices')

corr_matrices_unfiltered = []
corr_matrices = []


mask = ~np.isin(labeling, masked_labels)
clean_ts = cleanTS(sub, ses,bids_folder=bids_folder)
seed_ts = clean_ts[mask]

# filter out nodes that are not connected to the rest
correlation_measure = ConnectivityMeasure(kind='correlation')
graph = correlation_measure.fit_transform([seed_ts.T])[0] #correlation_matrix_noParcel
pd.DataFrame(graph).to_csv(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_unfiltered.csv')) # .shape = (18709, 18709)
#corr_matrices_unfiltered.append(pd.DataFrame(correlation_matrix))

cc = connected_components(graph)
mask_cc = cc[1] == 0 # all nodes in 0 belong to the largest connected component, check #-components in cc[0]
mask[mask == True] = mask_cc
seed_ts = clean_ts[mask]   

correlation_measure = ConnectivityMeasure(kind='correlation')
correlation_matrix = correlation_measure.fit_transform([seed_ts.T])[0]
pd.DataFrame(correlation_matrix).to_csv(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_onlyconncomp.csv')) # .shape = (18709, 18709)

#corr_matrices.append(pd.DataFrame(correlation_matrix))
print(sub, 'done')

09 done


In [5]:
# read in average connectivity file

#file_name_pq = op.join(target_folder,f'averageSub1-9_corrMatrix_{filtered}.parquet')
#cm_av = pd.read_parquet(file_name_pq) # 'suitable version of pyarrow or fastparquet is required for parquet support.'

filtered = 'unfiltered'

file_name_csv = op.join(target_folder,f'averageSub1-9_corrMatrix_{filtered}.csv')
cm_av = pd.read_csv(file_name_csv)

In [22]:
#from brainspace.gradient import GradientMaps
correlation_matrix = np.array(cm_av)
correlation_matrix = np.nan_to_num(correlation_matrix,nan=0.0)
gm = GradientMaps(n_components=2, random_state=0) # Default is 'dm' = DiffusionMaps
gm.fit(correlation_matrix)

ValueError: Array is not square.

In [23]:
np.shape(correlation_matrix)

(18715, 18716)

In [24]:
correlation_matrix

array([[ 0.00000000e+00,  1.00000000e+00,  1.51755189e-02, ...,
        -8.05217841e-03,  1.31862419e-02, -9.94345608e-03],
       [ 1.00000000e+00,  1.51755189e-02,  1.00000000e+00, ...,
         1.03626061e-02, -4.83360000e-03, -2.13592199e-02],
       [ 2.00000000e+00,  8.86704004e-02,  7.63482413e-02, ...,
        -1.13089147e-02,  1.08764632e-02,  1.30145838e-02],
       ...,
       [ 1.87120000e+04,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 1.87130000e+04,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 1.87140000e+04,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])